# 강의 04 · 실습 1 — 상태·노드·조건부 엣지 · (4) 고난도 I

## 1. 문제상황

- 사내 IT 헬프데스크의 접수 기록을 다시 보니, 요청을 긴급과 일반 둘로만 나누어서는 처리가 늦어지는 요청이 있습니다.
- 서버가 멈춰 여러 사람이 일을 못 하는 요청은 엔지니어를 지금 불러야 하고, 비밀번호를 잊은 요청은 안내문이면 끝납니다.
- 그런데 한 사람의 노트북이 켜지지 않는 요청은 둘 중 어디에도 맞지 않습니다. 엔지니어를 당장 부를 일은 아니지만, 안내문으로는 해결되지 않고 담당자가 자리를 방문해야 합니다.
- 담당자는 이런 요청을 일반으로 분류해 안내문을 보냈다가 다시 방문 일정을 잡는 일을 반복합니다. 요청자는 안내문을 받고도 해결되지 않아 두 번 문의합니다.

## 2. 문제와 목표

- **문제**: 긴급도를 둘로만 나누면 방문이 필요한 요청이 일반으로 분류되어, 안내문을 보낸 뒤 다시 방문 일정을 잡는 일이 반복됩니다.
- **목표**
  - 요청 한 건을 입력하면 프로그램이 긴급도를 셋 중 하나로 판정합니다.
    - 긴급도 셋: 긴급(서비스가 멈추었거나 여러 사람이 일을 못 함), 보통(한 사람의 장비나 계정이 고장 나 그 사람이 일을 못 함), 일반(안내문으로 스스로 처리할 수 있음)
  - 긴급이면 엔지니어 호출 메시지를, 보통이면 방문 일정 안내문을, 일반이면 자가 해결 안내문을 만듭니다.
  - 세 경우 모두 접수 기록까지 진행하는 처리 흐름을 만듭니다.
- **목표 달성 여부의 판정 기준**:
  - 긴급한 요청, 방문이 필요한 요청, 안내문으로 끝나는 요청을 한 건씩 입력했을 때,
  - 세 입력이 가운데에서 서로 다른 노드를 거치고 세 입력 모두 마지막에 기록 노드를 거치는 것을 실행 결과에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex01_s4_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 요청 본문(`ticket`), 긴급도 판정 결과(`level`), 요청자에게 나갈 글(`reply`), 기록 여부(`logged`) 키 네 개를 가지는 상태를 선언합니다.
    - 키 네 개 외의 값은 상태에 들어가지 않습니다.
2. **판정 노드를 만듭니다.**
    - triage 노드는 상태의 요청 본문을 읽고, 긴급·보통·일반 중 한 단어로 판정한 결과를 상태의 `level` 키에 씁니다.
    - 서비스가 멈추었거나 여러 사람이 일을 못 하면 긴급, 한 사람의 장비나 계정이 고장 나 그 사람이 일을 못 하면 보통, 안내문으로 요청자가 스스로 처리할 수 있으면 일반입니다.
3. **호출 메시지 노드를 만듭니다.**
    - escalate 노드는 상태의 요청 본문을 읽고, 담당 엔지니어를 호출하는 두 문장짜리 글을 상태의 `reply` 키에 씁니다.
4. **방문 일정 노드를 만듭니다.**
    - schedule 노드는 상태의 요청 본문을 읽고, 담당자가 요청자의 자리를 방문할 일정을 잡는 두 문장짜리 글을 상태의 `reply` 키에 씁니다.
    - 방문 전에 요청자가 준비할 것을 담습니다.
5. **안내문 노드를 만듭니다.**
    - answer 노드는 상태의 요청 본문을 읽고, 요청자가 스스로 처리할 방법을 알리는 두 문장짜리 글을 상태의 `reply` 키에 씁니다.
6. **기록 노드를 만듭니다.**
    - record 노드는 상태의 긴급도와 나갈 글을 읽어 접수 기록에 남기고, 상태의 `logged` 키에 기록 여부를 씁니다.
    - 이 실습에서 기록은 화면 출력으로 대신하며, 출력 줄은 「[기록] 긴급도=…」로 시작합니다.
7. **그래프에 노드를 등록합니다.**
    - 다섯 노드를 이름과 함께 그래프에 등록합니다.
8. **엣지를 연결합니다.**
    - START에서 triage로 가는 고정 엣지를 추가하고, triage 뒤에는 판정 결과를 보고 갈 곳을 고르는 조건부 엣지를 추가합니다.
    - 판정 결과가 긴급이면 escalate로, 보통이면 schedule로, 일반이면 answer로 갑니다.
    - escalate·schedule·answer 뒤에는 각각 record를, record 뒤에는 END를 고정 엣지로 연결합니다.
9. **그래프를 컴파일하고 실행합니다.**
    - 긴급한 요청, 방문이 필요한 요청, 안내문으로 끝나는 요청을 차례대로 넣고, 노드가 하나 끝날 때마다 상태의 어느 키가 채워졌는지 화면에 출력하고, 끝나면 최종 상태(긴급도와 기록 여부)를 한 줄로 출력합니다.
    - 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키를 선언합니다 | `class TicketState(TypedDict)` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `def triage(state) -> dict` | 2, 3, 4, 5, 6 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 함수에 이름을 붙여 등록합니다 | `StateGraph(TicketState)`, `add_node` | 7 |
| ④ 엣지 연결 | 노드 사이의 순서와 분기를 정합니다 | `add_edge`, `add_conditional_edges` | 8 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 입력을 넣어 실행합니다 | `compile()`, `stream()` | 9 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")


# 주어진 자료
TICKETS = [
    "사내 그룹웨어가 오전 9시부터 접속되지 않습니다. 부서 전체가 결재를 올리지 못하고 있습니다.",
    "제 노트북이 켜지지 않습니다. 오늘 오후에 보고서를 내야 하는데 쓸 수 있는 다른 PC가 없습니다.",
    "노트북 사내 와이파이 비밀번호를 잊어버렸습니다. 다시 알려주실 수 있을까요?",
]


### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. `TypedDict`로 선언한 네 개의 키가 이 그래프에서 오가는 데이터의 전부입니다.

In [ ]:
# 여기에 단계 ①(상태 정의)을 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3, 4, 5, 6)

- 노드는 상태를 인자로 받아 딕셔너리를 돌려주는 파이썬 함수입니다.
- 돌려준 딕셔너리가 상태의 해당 키를 덮습니다. 바뀐 키만 돌려주면 나머지 키는 그대로 남습니다.
- escalate·schedule·answer 노드는 상태의 같은 키(`reply`)에 씁니다. 셋 중 실행된 노드의 글만 상태에 남습니다.

In [ ]:
# 여기에 단계 ②(노드 함수 정의)를 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 7)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다. 여기서 붙인 이름은 뒤의 엣지 연결에서 그대로 쓰입니다.

In [ ]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 8)

`add_conditional_edges`의 판단 함수 `route`가 돌려줄 수 있는 이름이 셋입니다. 세 번째 인자로 넘기는 딕셔너리에 세 이름이 모두 들어 있어야 합니다. 판단 함수는 상태의 긴급도 판정 결과만 보고 이름을 고릅니다.

In [ ]:
# 여기에 단계 ④(엣지 연결)를 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 9)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `stream`은 노드가 하나 끝날 때마다 그 노드가 바꾼 부분을 내보냅니다. 아래에서는 세 종류의 요청을 차례대로 넣습니다.

`graph.stream(입력, stream_mode="updates")`로 부르면 노드가 하나 끝날 때마다 `{노드 이름: 바뀐 키}`가 나옵니다.


In [ ]:
# 여기에 단계 ⑤(컴파일과 실행)를 작성합니다.

## 7. 실행 결과 확인

실행 결과에서 다음 세 가지를 확인합니다.

1. 1번 접수(그룹웨어 장애)는 `triage`, `escalate`, `record`를, 2번 접수(노트북 고장)는 `triage`, `schedule`, `record`를, 3번 접수(와이파이 비밀번호)는 `triage`, `answer`, `record`를 차례대로 거칩니다. 가운데 노드만 셋이 다릅니다.
2. 세 접수 모두 `[기록]` 줄이 출력되고 마지막 줄의 `logged` 값이 `True`입니다. 분기는 셋이지만 record 노드는 모두 거쳤습니다.
3. `triage`가 돌려준 `level` 값이 긴급·보통·일반 셋 중 하나이고, 그 값과 가운데 노드의 이름이 요구사항 8의 대응대로 짝지어져 있습니다.